# Multimodel integration (Groq, Google)

In [38]:
import torch

import os
from dotenv import load_dotenv

load_dotenv(dotenv_path=r"config\.env")


# --- ROCm/CUDA device check ---
# ROCm exposes itself to PyTorch through the same torch.cuda API as NVIDIA CUDA,
if torch.cuda.is_available():
    device_name = torch.cuda.get_device_name(0)
    total_vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    print(f"GPU detected: {device_name} ({total_vram_gb:.1f} GB VRAM)")
else:
    print("WARNING: No GPU detected by torch — falling back to CPU. Check your ROCm/torch install.")


GPU detected: AMD Radeon RX 7900 XT (20.0 GB VRAM)


## Groq integration

In [39]:
from langchain.chat_models import init_chat_model

os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

model = init_chat_model(
        "openai/gpt-oss-20b",
        model_provider="groq",
    )

model

ChatGroq(metadata={'lc_versions': {'langchain-core': '1.6.0', 'langchain': '1.3.15'}}, profile={'name': 'GPT OSS 20B', 'release_date': '2025-08-05', 'last_updated': '2026-05-27', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 65536, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'structured_output': True, 'attachment': False, 'temperature': True}, client=<groq.resources.chat.completions.Completions object at 0x0000022D3FF6A4B0>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x0000022D3FF6C530>, model_name='openai/gpt-oss-20b', model_kwargs={}, groq_api_key=SecretStr('**********'))

In [40]:
response = model.invoke("Hello, how are you?")
response

AIMessage(content='Hello! I’m doing great—thanks for asking. How about you? Is there anything you’d like to chat about or any questions I can help with today?', additional_kwargs={'reasoning_content': 'User says "Hello, how are you?" We need to respond politely. The user is greeting. It\'s a normal conversation. We can respond with a friendly greeting, ask how they are. Use the guidelines: no policy violation. Just respond.'}, response_metadata={'token_usage': {'completion_tokens': 92, 'prompt_tokens': 77, 'total_tokens': 169, 'completion_time': 0.10128849, 'completion_tokens_details': {'reasoning_tokens': 50}, 'prompt_time': 0.003712841, 'prompt_tokens_details': None, 'queue_time': 0.149839054, 'total_time': 0.105001331}, 'model_name': 'openai/gpt-oss-20b', 'system_fingerprint': 'fp_3023a70d60', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a021af-ccfe-7663-b87e-92c837dcf85c-0', tool_calls=[], invalid_tool_calls=[], us

In [41]:
response.content

'Hello! I’m doing great—thanks for asking. How about you? Is there anything you’d like to chat about or any questions I can help with today?'

## Gemini integration

In [43]:
from langchain.chat_models import init_chat_model

os.environ["GOOGLE_API_KEY"] = os.getenv("GOOGLE_GENAI_API_KEY")

model = init_chat_model( # Method 1: Using init_chat_model with the model string
    "google_genai:gemini-3.5-flash-lite",
    )

response = model.invoke("Why do parrots have such colorful feathers?")
response.content

[{'type': 'text',
  'text': "Parrots are famous for their dazzling reds, blues, yellows, and greens. While they look like flying rainbows just to please our eyes, every vibrant color has a very specific evolutionary purpose. \n\nHere are the main reasons why parrots have such colorful feathers:\n\n### 1. Camouflage in the Rainforest\nThis surprises many people because bright colors sound like the opposite of camouflage. However, most parrots live in tropical rainforests, which are places of extreme contrast—bright sunlight piercing through dense green leaves creates deep shadows and brilliant spots of color. \n* **Green parrots** (like many Amazons and parakeets) blend in almost invisibly with the forest canopy when sitting still. \n* **Flash colors** (like red, yellow, and blue) actually help break up the bird’s silhouette, making it harder for predators like hawks and eagles to spot a distinct bird shape among the multicolored fruit, flowers, and dappled sunlight.\n\n### 2. Finding a

In [44]:
from langchain_google_genai import ChatGoogleGenerativeAI

model = ChatGoogleGenerativeAI( # Method 2: Using the ChatGoogleGenerativeAI class directly
    model="gemini-3.5-flash-lite",
)

response = model.invoke("Why do parrots have such colorful feathers?")
response.content

[{'type': 'text',
  'text': "Parrots have such brightly colored feathers for several important evolutionary reasons, mostly related to survival, communication, and their specific lifestyles. Unlike many other birds where males are colorful and females are dull, **both male and female parrots are usually brightly colored.** \n\nHere is a breakdown of why parrots wear such vibrant coats:\n\n### 1. Camouflage in the Rainforest Canopy\nTo the human eye standing on the ground, a bright green or red parrot in a tree looks like it would stand out immediately. However, in their natural habitat—the dappled, sun-drenched tropical rainforest—their colors actually act as camouflage. \n* **Green feathers** blend in seamlessly with the dense canopy leaves.\n* **Reds, yellows, and blues** break up the bird’s silhouette and mimic tropical flowers, fruits, or the bright patches of sunlight filtering through the trees. \n\n### 2. Species Recognition and Social Communication\nParrots are extremely social